In [ ]:
%%capture
%pip install zea

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"
os.environ["ZEA_DISABLE_CACHE"] = "1"

In [ ]:
from zea import init_device
from zea.data import load_file
from zea.visualize import set_mpl_style
from zea.display import to_8bit
from zea.ops import (
    Pipeline,
    Refocus,
    EnvelopeDetect,
    Normalize,
    LogCompress,
    Beamform,
)

import matplotlib.pyplot as plt

In [ ]:
init_device(verbose=False)
set_mpl_style()

path = "hf://zeahub/picmus/database/experiments/contrast_speckle/contrast_speckle_expe_dataset_rf/contrast_speckle_expe_dataset_rf.hdf5"

data, scan, probe = load_file(
    path=path,
    indices=[0],
    data_type="raw_data",
)
print(
    f"n_frames: {data.shape[0]}\n"
    f"n_tx: {data.shape[1]}\n"
    f"n_ax: {data.shape[2]}\n"
    f"n_el: {data.shape[3]}\n"
    f"n_ch: {data.shape[4]}"
)

In [ ]:
pipeline = Pipeline([
    Beamform(
        beamformer = "delay_and_sum",
        num_patches=200,
        enable_pfield=True,
    ),
    EnvelopeDetect(),
    Normalize(),
    LogCompress(),
])
parameters = pipeline.prepare_parameters(
    scan=scan,
    probe=probe,
)

inputs = {pipeline.key: data}
outputs = pipeline(**inputs, **parameters)
images = outputs[pipeline.key]

plt.imshow(to_8bit(images[0]), cmap="gray")

In [ ]:
Refocus_methods = [
    {"method": "adjoint", "param": None},
    {"method": "adjoint", "param": 0},
    {"method": "tikhonov", "param": 0.01},
    {"method": "tsvd", "param": 0.1}
]
fig, axes = plt.subplots(1, len(Refocus_methods), figsize=(20, 5))

for refocus_method in Refocus_methods:
    pipeline = Pipeline([
        Refocus(
            method=refocus_method["method"],
            param=refocus_method["param"],
        ),
        Beamform(
            beamformer = "delay_and_sum",
            num_patches=200,
            enable_pfield=True,
        ),
        EnvelopeDetect(),
        Normalize(),
        LogCompress(),
    ])
    parameters = pipeline.prepare_parameters(
        scan=scan,
        probe=probe,
    )

    inputs = {pipeline.key: data}
    outputs = pipeline(**inputs, **parameters)
    images = outputs[pipeline.key]

    axes[Refocus_methods.index(refocus_method)].imshow(to_8bit(images[0]), cmap="gray")
    axes[Refocus_methods.index(refocus_method)].set_title(
        f"{refocus_method['method']} (param={refocus_method['param']})"
    )
plt.savefig("refocus_methods_comparison.png")
plt.close()

![refocus_example](./refocus_methods_comparison.png)